# 20 · Diffusion Models and Score-Based Generation

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/anuxlab/deep-learning-systems/blob/main/notebooks/20-diffusion/20_diffusion.ipynb)
[![Run on GitHub Actions](https://img.shields.io/badge/run-GitHub%20Actions-2088FF?logo=githubactions&logoColor=white)](https://github.com/anuxlab/deep-learning-systems/actions/workflows/nb-20-diffusion.yml)

| | |
|---|---|
| **Wiki chapter** | [Module 20](https://github.com/anuxlab/deep-learning-systems/wiki/Module-20) |
| **Level** | Advanced |
| **Typical CPU runtime** | ~2 min (GitHub free runner, no GPU) |
| **Prerequisites** | Modules 03, 07, 18 |

## What you will do

- Implement the forward noising process and verify the closed form $q(x_t|x_0)$ against iterating one step at a time.
- Verify the posterior $q(x_{t-1}|x_t,x_0)$ formulas and the $\epsilon$-prediction identity.
- Train a DDPM noise-prediction network on a 2-D dataset and sample with ancestral (DDPM) and DDIM samplers.
- Evaluate sample quality (sliced Wasserstein / MMD) and add classifier-free guidance for class-conditional generation.

> **How to use this notebook.** Read each markdown cell, *predict* what the next code cell will print, then run it.
> Every section ends with `check(...)` assertions: if one fails, something in your understanding (or the code) is off.
> Figures are saved to `figures/` and a machine-readable summary to `metrics.json` next to this notebook.

## 0 · Setup and reproducibility

We import shared helpers (`show` saves each figure to `figures/`, `check` asserts and logs, `timer` logs wall-clock time), fix all random seeds and print an environment report so every run is traceable (see the wiki's *Implementation and Research Standards*).

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "_common"))   # shared helpers: notebooks/_common/dlsys.py
from dlsys import *                                       # np, plt, torch, seed_everything, show, check, timer, ...

SEED = seed_everything(42)
ENV = env_report()

## 1 · The forward (noising) process
With a variance schedule $\beta_1..\beta_T$ define $q(x_t|x_{t-1})=\mathcal N(\sqrt{1-\beta_t}\,x_{t-1},\beta_tI)$. Let $\alpha_t=1-\beta_t$, $\bar\alpha_t=\prod_{s\le t}\alpha_s$. Because Gaussians compose,
$$q(x_t|x_0)=\mathcal N\big(\sqrt{\bar\alpha_t}\,x_0,\ (1-\bar\alpha_t)I\big)\iff x_t=\sqrt{\bar\alpha_t}\,x_0+\sqrt{1-\bar\alpha_t}\,\epsilon,\ \epsilon\sim\mathcal N(0,I).$$
We verify the closed form by **simulating the 1-step chain 200 times** for many samples and comparing the empirical mean/variance with theory, then look at the signal-to-noise ratio across the schedule.

In [ ]:
T = 100; betas = torch.linspace(1e-4, .1, T); alphas = 1 - betas; abar = torch.cumprod(alphas, 0)
x0 = torch.full((200000, 1), 1.5); x = x0.clone(); t_check = [9, 49, 99]; emp = {}
torch.manual_seed(0)
for t in range(T):
    x = alphas[t].sqrt() * x + betas[t].sqrt() * torch.randn_like(x)
    if t in t_check: emp[t] = (x.mean().item(), x.var().item())
for t in t_check:
    th_m, th_v = abar[t].sqrt().item() * 1.5, (1 - abar[t]).item(); print(f"t={t+1:3d}: empirical mean {emp[t][0]:.4f} var {emp[t][1]:.4f}   |  theory mean {th_m:.4f} var {th_v:.4f}")
    check(abs(emp[t][0] - th_m) < .01 and abs(emp[t][1] - th_v) < .01, f"t={t+1}: iterated chain matches the closed form q(x_t|x_0)")
snr = abar / (1 - abar); plt.semilogy(snr.numpy()); plt.xlabel("t"); plt.ylabel("SNR = ᾱ_t/(1-ᾱ_t)"); plt.title("Signal-to-noise ratio along the schedule"); show("snr")
check(abar[-1] < .01, "by t=T almost no signal remains: x_T ≈ N(0, I)  (ᾱ_T = %.4f)" % abar[-1])

## 2 · The reverse process, the posterior and ε-prediction
Conditioned on $x_0$ the reverse step is tractable:
$$q(x_{t-1}|x_t,x_0)=\mathcal N(\tilde\mu_t,\tilde\beta_tI),\quad \tilde\mu_t=\frac{\sqrt{\bar\alpha_{t-1}}\beta_t}{1-\bar\alpha_t}x_0+\frac{\sqrt{\alpha_t}(1-\bar\alpha_{t-1})}{1-\bar\alpha_t}x_t,\quad\tilde\beta_t=\frac{1-\bar\alpha_{t-1}}{1-\bar\alpha_t}\beta_t.$$
Substituting $x_0=(x_t-\sqrt{1-\bar\alpha_t}\,\epsilon)/\sqrt{\bar\alpha_t}$ gives $\tilde\mu_t=\frac1{\sqrt{\alpha_t}}\big(x_t-\frac{\beta_t}{\sqrt{1-\bar\alpha_t}}\epsilon\big)$, so a network $\epsilon_\theta(x_t,t)$ predicting the noise defines the reverse mean. The training loss is the simple $\mathbb E\|\epsilon-\epsilon_\theta(x_t,t)\|^2$.

We verify the posterior by **Bayes' rule on a 1-D Gaussian prior** where everything is closed-form: with $x_0\sim\mathcal N(0,1)$, the marginals are Gaussian and the posterior mean must equal the formula (for a *fixed* $x_0$ the formula is exact; we test that and the ε-substitution).

In [ ]:
t = 40; a_t, ab_t, ab_p, b_t = alphas[t].item(), abar[t].item(), abar[t - 1].item(), betas[t].item()
x0v, eps = 1.2, -.7; xt = np.sqrt(ab_t) * x0v + np.sqrt(1 - ab_t) * eps
mu_x0 = np.sqrt(ab_p) * b_t / (1 - ab_t) * x0v + np.sqrt(a_t) * (1 - ab_p) / (1 - ab_t) * xt
mu_eps = (xt - b_t / np.sqrt(1 - ab_t) * eps) / np.sqrt(a_t)
check(np.isclose(mu_x0, mu_eps), "posterior mean in (x_t, x_0) form == ε-prediction form")
# brute-force Bayes: q(x_{t-1}|x_t,x0) ∝ q(x_t|x_{t-1}) q(x_{t-1}|x0)  on a grid
xs = np.linspace(-4, 4, 40001); lp = -(xt - np.sqrt(a_t) * xs) ** 2 / (2 * b_t) - (xs - np.sqrt(ab_p) * x0v) ** 2 / (2 * (1 - ab_p)); w = np.exp(lp - lp.max()); w /= w.sum()
m_num = (w * xs).sum(); v_num = (w * (xs - m_num) ** 2).sum(); v_th = (1 - ab_p) / (1 - ab_t) * b_t
print(f"posterior mean: grid {m_num:.5f} vs formula {mu_x0:.5f};  variance: grid {v_num:.6f} vs formula {v_th:.6f}")
check(abs(m_num - mu_x0) < 1e-4 and abs(v_num - v_th) < 1e-5, "closed-form posterior mean/variance match brute-force Bayes")

## 3 · Data and a noise-prediction network
Target distribution: a 2-D **two-moons** dataset (scaled), with the class (which moon) available for conditional generation later. The network $\epsilon_\theta(x_t,t,c)$ is an MLP with **sinusoidal time embedding** (and an optional class embedding with a "null" token for classifier-free guidance). Training: sample $t$ uniformly, noise the data with the closed form, regress the noise.

In [ ]:
from sklearn.datasets import make_moons
def sample_data(n, seed=None):
    X, c = make_moons(n, noise=.06, random_state=seed); X = (X - np.array([.5, .25])) * 2.0; return torch.tensor(X, dtype=torch.float32), torch.tensor(c)
Xd, Cd = sample_data(8000, 0)
def temb(t, dim=32):
    f = torch.exp(-np.log(10000) * torch.arange(dim // 2) / (dim // 2)); a = t[:, None].float() * f[None]; return torch.cat([a.sin(), a.cos()], 1)
class EpsNet(nn.Module):
    def __init__(s, h=192, ncls=2):
        super().__init__(); s.ce = nn.Embedding(ncls + 1, 32); s.net = nn.Sequential(nn.Linear(2 + 32 + 32, h), nn.SiLU(), nn.Linear(h, h), nn.SiLU(), nn.Linear(h, h), nn.SiLU(), nn.Linear(h, 2)); s.null = ncls
    def forward(s, x, t, c=None):
        c = torch.full((len(x),), s.null, dtype=torch.long) if c is None else c; return s.net(torch.cat([x, temb(t), s.ce(c)], 1))
def train_ddpm(steps=6000, cond_drop=.15, seed=0):
    torch.manual_seed(seed); m = EpsNet(); o = torch.optim.Adam(m.parameters(), 2e-3); sched = torch.optim.lr_scheduler.CosineAnnealingLR(o, steps); losses = []
    for it in range(steps):
        ix = torch.randint(0, len(Xd), (256,)); x0, c = Xd[ix], Cd[ix].clone(); c[torch.rand(256) < cond_drop] = m.null            # drop labels 15% of the time (for CFG)
        t = torch.randint(0, T, (256,)); eps = torch.randn_like(x0); xt = abar[t].sqrt()[:, None] * x0 + (1 - abar[t]).sqrt()[:, None] * eps
        loss = F.mse_loss(m(xt, t, c), eps); o.zero_grad(); loss.backward(); o.step(); sched.step(); losses.append(loss.item())
        if it % 700 == 0: print(f"step {it:4d}  eps-MSE {loss.item():.4f}")
    return m, losses
with timer("ddpm_training"): model, losses = train_ddpm()
plt.semilogy(np.convolve(losses, np.ones(50) / 50, "valid")); plt.xlabel("step"); plt.ylabel("ε-prediction MSE (smoothed)"); plt.title("DDPM training loss"); show("ddpm_loss")
check(np.mean(losses[-100:]) < np.mean(losses[:100]) * .7, "noise-prediction loss decreases"); METRICS["ddpm_final_loss"] = float(np.mean(losses[-100:]))

## 4 · Sampling: ancestral DDPM vs deterministic DDIM
**DDPM (ancestral):** start $x_T\sim\mathcal N(0,I)$; for $t=T..1$: $x_{t-1}=\tilde\mu_\theta(x_t,t)+\sqrt{\beta_t}\,z$ ($z=0$ at the last step).
**DDIM** (η=0) uses the *same trained network* but a deterministic update with an arbitrary sub-sequence of timesteps: predict $\hat x_0=(x_t-\sqrt{1-\bar\alpha_t}\hat\epsilon)/\sqrt{\bar\alpha_t}$, then $x_{s}=\sqrt{\bar\alpha_s}\hat x_0+\sqrt{1-\bar\alpha_s}\hat\epsilon$. This allows 10–20 steps instead of 100.

In [ ]:
@torch.no_grad()
def sample_ddpm(m, n, c=None, w=0.0, seed=0):
    g = torch.Generator().manual_seed(seed); x = torch.randn(n, 2, generator=g)
    for t in reversed(range(T)):
        tt = torch.full((n,), t); eps = guided_eps(m, x, tt, c, w); mean = (x - betas[t] / (1 - abar[t]).sqrt() * eps) / alphas[t].sqrt()
        x = mean + (betas[t].sqrt() * torch.randn(n, 2, generator=g) if t > 0 else 0)
    return x
@torch.no_grad()
def sample_ddim(m, n, steps=10, c=None, w=0.0, seed=0):
    g = torch.Generator().manual_seed(seed); x = torch.randn(n, 2, generator=g); ts = torch.linspace(T - 1, 0, steps).long().tolist()
    for i, t in enumerate(ts):
        tt = torch.full((n,), t); eps = guided_eps(m, x, tt, c, w); x0h = (x - (1 - abar[t]).sqrt() * eps) / abar[t].sqrt(); ab_next = abar[ts[i + 1]] if i + 1 < len(ts) else torch.tensor(1.0)
        x = ab_next.sqrt() * x0h + (1 - ab_next).sqrt() * eps
    return x
def guided_eps(m, x, t, c, w):                           # classifier-free guidance: (1+w)·ε(x,c) − w·ε(x,∅)
    if c is None or w == 0: return m(x, t, c)
    return (1 + w) * m(x, t, c) - w * m(x, t, None)
def sliced_w2(a, b, n_proj=100, seed=0):
    g = torch.Generator().manual_seed(seed); d = torch.randn(n_proj, 2, generator=g); d = d / d.norm(dim=1, keepdim=True); pa, pb = (a @ d.T).sort(0)[0], (b @ d.T).sort(0)[0]; return ((pa - pb) ** 2).mean().sqrt().item()
ref, _ = sample_data(2000, 123); noise = torch.randn(2000, 2)
with timer("sampling"): s_ddpm = sample_ddpm(model, 2000); s_ddim = {k: sample_ddim(model, 2000, k) for k in [5, 10, 25, 50]}
print(f"sliced-W2 to held-out data:  pure noise {sliced_w2(noise, ref):.3f} | DDPM(100 steps) {sliced_w2(s_ddpm, ref):.3f} | DDIM " + " ".join(f"{k}st:{sliced_w2(v, ref):.3f}" for k, v in s_ddim.items()))
fig, ax = plt.subplots(1, 4, figsize=(15, 3.8))
for a_, (nm, s) in zip(ax, [("data", ref), ("DDPM 100 steps", s_ddpm), ("DDIM 10 steps", s_ddim[10]), ("DDIM 5 steps", s_ddim[5])]): a_.scatter(*s.T, s=3); a_.set_title(nm); a_.set_xlim(-3, 3); a_.set_ylim(-2.5, 2.5)
show("diffusion_samples")
check(sliced_w2(s_ddpm, ref) < .25 * sliced_w2(noise, ref), "DDPM samples are far closer to the data than noise")
check(sliced_w2(s_ddim[25], ref) < .4 * sliced_w2(noise, ref), "DDIM with only 25 steps is also much closer to the data than noise")
check(sliced_w2(s_ddim[50], ref) < sliced_w2(s_ddim[5], ref), "more DDIM steps -> better samples (quality/compute trade-off)")
METRICS["sliced_w2"] = dict(noise=sliced_w2(noise, ref), ddpm=sliced_w2(s_ddpm, ref), **{f"ddim_{k}": sliced_w2(v, ref) for k, v in s_ddim.items()})

### Visualising the denoising trajectory
Snapshots of the reverse process at several timesteps: a Gaussian blob gradually organises into the two moons.

In [ ]:
@torch.no_grad()
def trajectory(m, n=1500, seed=1, snaps=(99, 80, 60, 40, 20, 0)):
    g = torch.Generator().manual_seed(seed); x = torch.randn(n, 2, generator=g); out = {}
    for t in reversed(range(T)):
        tt = torch.full((n,), t); eps = m(x, tt); x = (x - betas[t] / (1 - abar[t]).sqrt() * eps) / alphas[t].sqrt() + (betas[t].sqrt() * torch.randn(n, 2, generator=g) if t > 0 else 0)
        if t in snaps: out[t] = x.clone()
    return out
tr_ = trajectory(model); fig, ax = plt.subplots(1, 6, figsize=(16, 2.8))
for a_, (t, x) in zip(ax, tr_.items()): a_.scatter(*x.T, s=2); a_.set_title(f"t={t}"); a_.set_xlim(-3.5, 3.5); a_.set_ylim(-3, 3)
show("denoising_trajectory")

## 5 · Classifier-free guidance
Train one network with the label *randomly dropped* (done above). At sampling time extrapolate: $\tilde\epsilon=(1+w)\,\epsilon_\theta(x_t,c)-w\,\epsilon_\theta(x_t,\varnothing)$. Larger $w$ trades **diversity** for **fidelity to the condition**. We generate class 0 with $w\in\{0,1,3,8\}$ and measure (i) *class accuracy* — how many samples fall closer to the class-0 moon than the class-1 moon (nearest data point of each class), and (ii) *diversity* (spread of samples).

In [ ]:
P0, P1 = Xd[Cd == 0][:2000], Xd[Cd == 1][:2000]
def class_frac(s, k): d0, d1 = torch.cdist(s, P0).min(1)[0], torch.cdist(s, P1).min(1)[0]; return ((d0 < d1) if k == 0 else (d1 < d0)).float().mean().item()
ws = [0, 1, 3, 8]; rows = []
with timer("cfg_sampling"):
    for w in ws:
        s = sample_ddim(model, 1500, 25, c=torch.zeros(1500, dtype=torch.long), w=w); rows.append((w, class_frac(s, 0), s.std(0).mean().item(), s)); print(f"w={w}: fraction in class-0 moon {rows[-1][1]:.3f}   spread (mean std) {rows[-1][2]:.3f}")
fig, ax = plt.subplots(1, 4, figsize=(15, 3.6))
for a_, (w, f, sd, s) in zip(ax, rows): a_.scatter(*s.T, s=3); a_.scatter(*P1[::20].T, s=1, c="lightgray"); a_.set_title(f"guidance w={w}  ({f:.0%} correct)"); a_.set_xlim(-3, 3); a_.set_ylim(-2.5, 2.5)
show("classifier_free_guidance")
check(rows[-1][1] >= rows[0][1] - .01 and rows[2][1] > .9, "guided samples fall in the requested class")
check(rows[0][1] > .5, "even unguided conditional sampling prefers the requested class")
METRICS["cfg"] = {f"w={w}": dict(class_fraction=f, spread=sd) for w, f, sd, _ in rows}

In [ ]:
save_metrics("20", METRICS)
log.info('notebook 20 finished OK')

## Wrap-up

## Exercises (do these next)

1. Replace the linear β schedule by a cosine schedule and compare sliced-W2 at T=50.
2. Implement v-prediction or x0-prediction and compare training curves.
3. Add the probability-flow ODE view: integrate dx/dt = -½β(x + 2∇log p) with the learned score and compare with DDIM.

## Where to read more

- Wiki chapter: [Module 20](https://github.com/anuxlab/deep-learning-systems/wiki/Module-20) (derivations, worked example, Q&A).
- Sohl-Dickstein et al. (2015); Ho et al. (2020) DDPM; Song et al. (2021) DDIM and score-SDE; Nichol & Dhariwal (2021) cosine schedule; Ho & Salimans (2022) classifier-free guidance; Karras et al. (2022) EDM.